In [35]:
import csv # for writing dataframes to csv
import random # for making a random choice
import os # for scanning directories
import itertools
from collections import Counter
from string import ascii_uppercase


import kintypes as kt # bringing large lists of kin types into the namespace
import math # for calculating logs
import pandas as pd
import numpy as np
import scipy.stats
import statistics
import re
from tqdm import tqdm

from Levenshtein import distance as lvs_dist

# Internal co-selection

Internal co-selection refers to a process of kin term evolution whereby terminological changes in one part of the paradigm co-occur with changes in related parts of the paradigm, increasing the predictive structure of the paradigm.

In this notebook, we will build simulations to investigate the robustness of this tendency cross-linguistically, using data from Kinbank, a global database of kin terminology. 

We will measure predictive structure between two generations as the symmetric conditional entropy between each of those generations given the other, i.e. H(X|Y) + H(Y|X).


### 1. Extract kin terminology from Kinbank

First, let's actually load our data in.

In [36]:
kinbank = pd.read_csv('../../data/kinbank.csv')

# get all glottocodes
all_glottocodes = list(set(kinbank['Glottocode']))

['siri1273',
 'slav1253',
 'caro1242',
 'kumy1244',
 'nepa1254',
 'wail1244',
 'nort2942',
 'siee1239',
 'umpi1239',
 'kumb1268',
 'urdu1245',
 'dehu1237',
 'siwa1245',
 'pohn1238',
 'eyak1241',
 'ling1263',
 'beka1241',
 'usar1243',
 'kama1365',
 'tido1248',
 'bign1238',
 'wang1288',
 'here1253',
 'suru1262',
 'lauj1238',
 'mido1240',
 'urav1235',
 'nyam1271',
 'bung1269',
 'bier1244',
 'mara1378',
 'mari1426',
 'urip1239',
 'wels1247',
 'toca1235',
 'budz1238',
 'tswa1253',
 'ital1282',
 'renn1242',
 'tami1289',
 'tupi1274',
 'otoo1241',
 'limi1242',
 'aman1265',
 'ponc1241',
 'wyan1247',
 'taus1251',
 'duma1253',
 'yulp1239',
 'hunz1247',
 'mber1257',
 'chip1262',
 'raro1241',
 'yami1254',
 'tuam1242',
 'cent2142',
 'para1312',
 'touo1238',
 'udmu1245',
 'nort2836',
 'tiko1237',
 'toho1245',
 'wata1253',
 'cota1241',
 'panj1256',
 'taji1246',
 'saar1237',
 'hiww1237',
 'ngon1269',
 'mali1284',
 'movi1243',
 'nort2944',
 'band1339',
 'gimi1243',
 'east1436',
 'yugu1249',
 'enap1235',

`get_kin_terms()` uses a language's Glottocode to extract a dictionary of kin terms for that language from thr `kinbank` data.

In [37]:
def get_kin_terms(glottocode):
    language_data = kinbank[kinbank['Glottocode'] == glottocode]

    kinterms = {}

    for row in range(len(language_data)): # for each row of the data
        term = list(language_data['Form'])[row] # get the kin term
        relative = list(language_data['Parameter_ID'])[row] # and kin type stored in that row
        try:
            kinterms[relative] = term.lower() # add them to dict
        except:
            pass

    return kinterms

### 2. Pair up parent and child terms.

We're interested in the predictive structure between the kin terms in Generation 0 (Ego's generaiton) and Generation +1 (Ego's parents' generation).

We want create a data structure that pairs up parent types with the corresponding child types. This is because we're interested in whether kinship systems maintain patterns of terminological distinctions and mergers across these two generations, so we will need to know which parent terms 'go with' which child terms.

`get_pairs()` takes a kinship system as input, and outputs a list of tuples. The first element in the tuple is the parent term, the second is the corresponding child term. 

But for our calculations, we'll still need to know which terms belong to which generation. Luckily, we know that the 0th element in each tuple is from Ego's parents' generation and the 1st element is from Ego's generation. So we can happily split these tuples down the middle and populate two lists with the terms. `split_pairs()` takes a list of pairs and sorts it into terms that belong to Ego's generation and terms that belong to Ego's parents' generation.

In [38]:
def get_pairs(ks: dict, pairs:list) -> list:
    pairs_of_terms = []
    pairs_of_relatives = []

    for pair in pairs:
        if pair[0] in ks and pair[1] in ks:
            pairs_of_terms.append((ks[pair[0]],ks[pair[1]]))
            pairs_of_relatives.append((pair[0],pair[1]))
                            
    return pairs_of_terms,pairs_of_relatives

In [39]:
def split_pairs(pairs: list) -> list:
    gn = []
    gn1 = []
    for pair in pairs:
        gn.append(pair[1])
        gn1.append(pair[0])
    
    return gn,gn1

### 3. Calculate probabilities

To calculate entropy, we need a probability distribution over the terms in one single generation of a kinship system. So let's start with a function that can calculate the probability of a particular term.

Given a term and the full list of terms in the same generation, `probability()` counts how many times that term exists in `generation` and divides that by the total length of `generation`.

To calculate **conditional entropy** between the two generations of our system, we will need not only the probabilities of terms in a generation, but also the **joint probabilities** of every pair of terms across those two generations. In other words, we need to calculate the probabilities of our `get_pairs` output.

Given two terms, `joint_probability()` counts how many pairs made of those two terms exist in `pairs`, then divides that by the total length of `pairs`.

In [40]:
def probability(term: str, generation: list) -> float:
    return generation.count(term)/len(generation)

In [41]:
def joint_probability(term1: str, term2: str, pairs:list) -> float:
    pair = (term1,term2)
    
    return pairs.count(pair)/len(pairs)

In [16]:
def joint_probability_distribution(ks):
    
    terms,relatives = get_pairs(ks,kt.ics_pairs)
    
#     print(terms)
    
    parents = ['mM','fM','mF','fF']
        
    probs = {k:v for (k,v) in zip(terms, [0 for i in range(len(terms))])}  
    
    parent_pairs = 0
    parent_terms = []
    
    for pair in range(len(relatives)):
        if relatives[pair][0] in parents:
            parent_pairs += 1
            parent_terms.append(terms[pair][0])

    for i in range(len(terms)):
        if relatives[i][0] not in parents:
            probs[terms[i]] += 1/(len(relatives) - (parent_pairs/2))
        else:
            probs[terms[i]] += (1/(len(relatives) - (parent_pairs/2))) / 2

    return probs

### 4. Calculating symmetric conditional entropy 

Conditional entropy of Y given X is defined as

$$
H(Y|X) = -\sum_{x \in X,y \in Y}p(x,y) log_2 {p(x,y) \over p(x)}
$$

or in English, the inverse sum over two distributions Y and X of the probability of each y * the log probability of each y given x.

Conditional entropy is the amount of information needed to describe the outcome of a random variable Y given that we already know the value of another random variable X.

To calculate it, we need the joint probability of each pair (given by `joint_probability()`) and the probability of one member of that pair (given by `probability()`). We can then calculate the conditional probability of parent term given child term as the joint probability of those terms over the probability of the parent term.

`conditional_entropy()` iterates over all pairs to output the conditional entropy of Ego's generation given Ego's parents' generation.


In [17]:
# calculate the entropy of terms in generation X given the terms in generation Y
def conditional_entropy(ks,x,y):
    """Calculate the summed entropy of all pairs of kin terms in the system. 
    X and Y indicate which variable conditions which."""
    entropy = 0
    
    pairs_of_terms,relatives = get_pairs(ks,kt.ics_pairs)
    g0,g1 = split_pairs(pairs_of_terms)
    
    probabilities = joint_probability_distribution(ks)
    
    for parent,child in set(pairs_of_terms): # for each pair of terms; x = parent, y = child
        p_xy = probabilities[parent,child] # calculate their joint probability
        p_ch = probability(child,g0) # calculate the probability of the child term in its generation
        p_par = probability(parent,g1) # calculate the probability of the parent term in its generation
        
        if x == 0: # if we want ce of child term given parent term (H(child|parent)):
            entropy += p_xy * math.log2(p_xy/p_par)
        if x == 1: # if we want ce of parent term given child term (H(parent|child)):
            entropy += p_xy * math.log2(p_xy/p_ch)
            
    return round(-entropy,5)

In [15]:
conditional_entropy(get_kin_terms('stan1293'),0,1)

0.13697

Since our measure takes the sum of conditional entropy 'in both directions', we'll need a function that calculates that too.

In [46]:
def predictive_structure(ks):
    """Calculate the sum of H(X|Y) and H(Y|X)."""
    
    hxy = conditional_entropy(ks,0,1)
    hyx = conditional_entropy(ks,1,0)
    
    return hxy + hyx

In [19]:
predictive_structure(get_kin_terms('stan1293'))

1.07394

For each language, calculate its symmetric conditional entropy and save to a dataframe.

In [4]:
def calculate_PS(filename):
    df = []
    codes = []
            
    for code in tqdm(all_glottocodes):

        language = list(kinbank[kinbank['Glottocode'] == code]['Name'])[0] # get the language name
        family = list(kinbank[kinbank['Glottocode'] == code]['Family'])[0] # get the family 

        if code not in codes:
            codes.append(code)

            ks = get_kin_terms(code)

            pairs,relatives = get_pairs(ks,kt.ics_pairs)

            if pairs: # if pairs is not empty

                results = {}
                results['language'] = language
                results['language_family'] = family
                results['code'] = code
                results['simulation'] = 'N'
                results['predictive_structure'] = predictive_structure(ks)

                df.append(results)
        
    pd.DataFrame(df).to_csv('data/' + filename + '.csv',index=False)
    
    return pd.DataFrame(df)

In [5]:
# calculate_PS('typological_data')

## Simulating kinship systems

To investigate whether kinship systems have more predictive structure than chance, we can build a random baseline for each language to serve as a point of comparison.

To do this, we will take each language in our dataset, and randomly scramble which terms go with which relatives (within generations). This will randomise the syncretisms within the paradigm, breaking any predictable structure built by the internal co-selection process, while maintaining the amount of variation across the system overall.
 
We take the following steps:

1. Extract the kinship system of a language from kinbank.
2. Filter the two generations we are interested in.
3. Randomly reassign the kinship terms to new types.
4. Repeat the process 1000 times for each language.

We already have the infrastructure for the first two! `get_kin_terms()`,  `get_pairs()` and `split_pairs()` will do this for us. So let's skip to 3, and write a function that randomises which terms form pairs, assuming that we have already extracted the kinship system and filtered the relevant pairs.

### 3. Randomly rearrange kin terms

`kintypes.py` also contains a list of which kin types are in which generation, which we can use to split a kinship system by generation.

In [7]:
def split_ks(ks):
    """Split the kinship system into two dictionaries where keys are relatives and values are terms.
    One dict is the generation N terms, one is the generation N+1 terms."""
    gn = {}
    gn1 = {}
    for entry in ks:
        if entry in kt.generation_n:
            gn[entry] = ks[entry]
        elif entry in kt.generation_n1:
            gn1[entry] = ks[entry]
        else:
            pass

    return gn,gn1        

def shuffle_ks(ks):
    """Split the kinship system, shuffle the Generation N terms, then reassign keys to values."""
    gn_terms = []
    gn,gn1 = split_ks(ks)
    
    new_ks = {}
    
    for term in gn:
        gn_terms.append(gn[term])

    random.shuffle(gn_terms)
    
    for i in range(len(gn)):
        key = list(gn.keys())[i]
        new_ks[key] = gn_terms[i]
        

    return {**new_ks,**gn1}


Finally, let's run the simulation - shuffle each kinship system 1000 times, measure the predictive structure of each permutation, and calculate a z-score per language that tells us how extreme the true predictive structure of the system is.

In [18]:
def simulate_PS(filename: str, filename2: str, times: int):
    df = []
    z_df = []
    codes = []
        
    for code in tqdm(all_glottocodes):
        language = list(kinbank[kinbank['Glottocode'] == code]['Name'])[0] # get the language name
        family = list(kinbank[kinbank['Glottocode'] == code]['Family'])[0] # get the family 


        if code not in codes:
            codes.append(code)

            ks = get_kin_terms(code)

            true_value = predictive_structure(ks)
            
            simulated_values = []

            for i in range(times):
                sim = shuffle_ks(ks)
                pairs,relatives = get_pairs(sim,kt.ics_pairs)
                
                if pairs:
                    
                    ps = predictive_structure(sim)
                    simulated_values.append(ps)
                    
                    results = {}
                    results['language'] = language
                    results['language_family'] = family
                    results['code'] = code
                    results['simulation'] = 'Y'
                    results['predictive_structure'] = ps
                    results['true_value'] = true_value

                    df.append(results)
            
            if pairs:
                mean = np.mean(simulated_values)
                sd = np.std(simulated_values)

                z_results = {}
                z_results['language'] = language
                z_results['language_family'] = family
                z_results['code'] = code
                z_results['true_value'] = true_value
                z_results['simulated_mean'] = mean
                z_results['simulated_sd'] = sd
                z_results['z'] = (true_value - mean) / sd

                z_df.append(z_results)
                
    
    pd.DataFrame(df).to_csv('data/' + filename + '.csv',index=False)
    pd.DataFrame(z_df).to_csv('data/' + filename2 + '.csv',index=False)

    
    return pd.DataFrame(z_df)

In [9]:
# simulate_PS('simulated_data','predictive_structure_z',1000)

## Edit Distance

Predictive structure may be an artefact of compositionality - where kin terms are compositional between parent and child terms, as in Kurdish, predictive structure follows naturally. Here, we test whether kin term compositionality and predictive structure are correlated.

We measure compositionality as the average normalised Levenshtein edit distance between parent and child terms. To calculate this, we measure the edit distance between each pair of parent and child terms.

Note: we are not just comparing all labels for all meanings, but rather all labels for all categories! This prevents languages being penalised for being compositional but not having e.g. all unique labels across aunts and uncles.

So the first thing we need to do is work out which categories each language has, and who are the children of which category members.


In [10]:
def category_based_pairs(g1):
    
    # work out which g1 relatives share a term
    
    relatives = list(g1.keys())
    terms = list(g1.values())

    categories = []

    for term in terms:
        cat = []
        indices = [index for index, element in enumerate(terms) if element == term]
        for index in indices:
            cat.append(relatives[index])
        categories.append(cat)

    categories = [list(i) for i in set(map(tuple, categories))]
    
    # work out which children should share a term on this basis
    
    child_cats = []

    for cat in categories:
        new_category = []
        for individual in cat:
            for pair in kt.ics_pairs:
                if pair[0] == individual:
                    new_category.append(pair[1])
        child_cats.append(new_category)
        
    # and make a new list of parent-child pairs that should have equal semantic distance
    
    new_pairs = []
    
    for i in range(len(categories)):
        new_pairs += list(itertools.product(categories[i], child_cats[i]))
        
    return new_pairs


Now we need a function that spits out an array of edit distances for our language.

In [11]:
def edit_distance(g0,g1):
    edit_dists = []

    for relative1 in g0:
        term1 = g0[relative1]
        for relative2 in g1:
            data = {}
            term2 = g1[relative2]
            
            if relative1 == relative2:
                pass
            
            else:
            
                if len(term1) > 0 and len(term2) > 0:
                    dist = lvs_dist(term1,term2)/len(max(term1,term2))
                    edit_dists.append(dist)
                else:
                    pass

    return edit_dists

Together with our kinship system shuffling infrastructure, we can calculate a z-score for these edit distance measures - do kinship systems have extremely low edit distance? In our analysis, we look at how this corresponds with extremely low predictive structure.

In [13]:
def simulate_edit_distance(filename,times):
    df = []
    
    codes = []
        
    for code in tqdm(all_glottocodes):

        language = list(kinbank[kinbank['Glottocode'] == code]['Name'])[0] # get the language name
        family = list(kinbank[kinbank['Glottocode'] == code]['Family'])[0] # get the family 

        data = {}

        if code not in codes:
            codes.append(code)

            ks = get_kin_terms(code)
            pairs,relatives = get_pairs(ks)
            
            avgs = []

            for pair in pairs:
                try:
                    ed = lvs_dist(pair[0],pair[1]) / max(len(pair[0]),len(pair[1]))
                    avgs.append(ed)
                except:
                    continue

            try:            
                true_mean = np.mean(avgs)

            except:
                continue

            sim_avgs = []
            for i in range(times):
                sim = shuffle_ks(ks)
                sim_pairs = get_pairs(sim)[0]

                distances = []
                for pair in sim_pairs:
                    try:
                        sim_ed = lvs_dist(pair[0],pair[1]) / max(len(pair[0]),len(pair[1]))
                        distances.append(sim_ed)
                    except:
                        continue
                        
                sim_avgs.append(np.mean(distances))


            sim_mean = np.mean(sim_avgs)
            sd = np.std(sim_avgs)

            z = (true_mean - sim_mean) / sd

            data['language'] = language
            data['family'] = family
            data['code'] = code
            data['true_value'] = true_mean
            data['simulated_mean'] = sim_mean
            data['sd'] = sd
            data['z'] = z

            df.append(data)
                
    pd.DataFrame(df).to_csv('../data/new_measure/' + filename + '.csv',index=False)

    return pd.DataFrame(df)

In [14]:
# simulate_edit_distance('edit_distance',1000)

# Reviewer concerns

Reviewer 2 suggested that string length might be affecting our measure of compositionality. Here, we calculate the edit distance between randomly generated strings of the same length as in the real languages, so we can check that the edit distance of randomly generated strings is sufficiently different from the edit distance of the actual languages in our dataset.

In [ ]:
def replace_terms_with_strings(ks):
    
    new_ks = {}
        
    for key in ks:
        value = ks[key]
        new_ks[key] = ''.join([random.choice(ascii_uppercase) for i in range(len(value))])

    return new_ks

In [12]:
def random_edit_distance(filename,times):
    df = []
    
    codes = []
        
    for code in tqdm(all_glottocodes):

        language = list(kinbank[kinbank['Glottocode'] == code]['Name'])[0] # get the language name
        family = list(kinbank[kinbank['Glottocode'] == code]['Family'])[0] # get the family 

        data = {}

        if code not in codes:
            codes.append(code)

            true_ks = get_kin_terms(code)
            ks = replace_terms_with_strings(true_ks)
            pairs,relatives = get_pairs(ks)
            
            avgs = []

            for pair in pairs:
                try:
                    ed = lvs_dist(pair[0],pair[1]) / max(len(pair[0]),len(pair[1]))
                    avgs.append(ed)
                except:
                    continue

            try:            
                true_mean = np.mean(avgs)

            except:
                continue

            sim_avgs = []
            for i in range(times):
                sim = shuffle_ks(ks)
                sim_pairs = get_pairs(sim)[0]

                distances = []
                for pair in sim_pairs:
                    try:
                        sim_ed = lvs_dist(pair[0],pair[1]) / max(len(pair[0]),len(pair[1]))
                        distances.append(sim_ed)
                    except:
                        continue
                        
                sim_avgs.append(np.mean(distances))


            sim_mean = np.mean(sim_avgs)
            sd = np.std(sim_avgs)

            z = (true_mean - sim_mean) / sd

            data['language'] = language
            data['family'] = family
            data['code'] = code
            data['true_value'] = true_mean
            data['simulated_mean'] = sim_mean
            data['sd'] = sd
            data['z'] = z

            df.append(data)
                
#     pd.DataFrame(df).to_csv('data/' + filename + '.csv',index=False)

    return pd.DataFrame(df)

In [ ]:
# random_edit_distance('edit_distance_random_strings',1000)

### Appendix D

Reviewer 2 suggested that we check what this analysis looks like with the restricted number of kin types that were present in the experiment. It may be that our results are not so strong compared to the true data even when we control for the number of kin types.

That said, real languages are still free to vary much more than our experiment data, since the terms for some individuals are fixed in the experiment.

In [33]:
exp_pairs = [

# parents and siblings

('mM','mB'), ('mF','mB'),
('mM','mZ'), ('mF','mZ'),

# nuncles and cousins

# mother's brother

('mMeB','mMBS'),('mMeB','mMBD'),

# mother's sister
    
('mMeZ','mMZS'),('mMeZ','mMZD'),
    
# father's brother
    
('mFeB','mFBS'),('mFeB','mFBD'),
    
# father's sister

('mFeZ','mFZS'),('mFeZ','mFZD')

]

We need to update our `split_ks` functions slightly so that they use this smaller version of the meaning space. 

**We also need to make sure `get_pairs()` is using the `exp_pairs` array (inside the `simulate_PS()`, `joint_probability_distribution()` and `conditional_entropy()` functions).** This is set to `kt.ics_pairs` for the original analysis.

In [15]:
gn = ['mB','mZ','mMBS','mMBD','mMZS','mMZD','mFBS','mFBD','mFZS','mFZD']
gn1 = ['mM','mF','mMeB','mMeZ','mFeB','mFeZ']

def split_ks(ks):
    g0 = {}
    g1 = {}
    for entry in ks:
        if entry in gn:
            g0[entry] = ks[entry]
        elif entry in gn1:
            g1[entry] = ks[entry]
        else:
            pass
        
#     print(g0,g1)

    return g0,g1    

In [59]:
# simulate_PS('small_sim','small_sim_z',1000)

  0%|▎                                                                                | 4/1127 [00:00<01:26, 13.03it/s]C:\Users\s1604058\AppData\Local\Temp\ipykernel_3516\950065671.py:52: RuntimeWarning: invalid value encountered in scalar divide
  z_results['z'] = (true_value - mean) / sd
100%|██████████████████████████████████████████████████████████████████████████████| 1127/1127 [01:35<00:00, 11.75it/s]


,language,language_family,code,true_value,simulated_mean,simulated_sd,z
0,Sirionó,Tupian,siri1273,0.94287,1.974595,0.284843,-3.622085
1,Kumyk,Turkic,kumy1244,1.07394,1.890741,0.161912,-5.044737
2,Nepali,Indo-European,nepa1254,0.33008,0.330080,0.000000,NaN
3,Eel River Athabaskan,NaN,wail1244,2.00000,1.354000,0.935245,0.690728
4,North Slavey,NaN,nort2942,3.00000,2.162000,0.541993,1.546147
...,...,...,...,...,...,...,...
733,Mwotlap,Austronesian,motl1237,3.00000,2.184000,0.525494,1.552824
734,Eastern Bolivian Guaraní,Tupian,east2555,1.87394,2.272715,0.133060,-2.996948
735,Northern Mansi,Uralic,mans1258,1.87394,2.271513,0.134958,-2.945906
736,Waropen,Austronesian,waro1242,1.00000,1.000000,0.000000,NaN


### Appendix C: Access to data affects predictiveness score

Checking whether languages with fewer kin types in their dataset

In [19]:
def count_data(ks):
    relatives = []
    for relative in ks:
#         print(relative)
        if relative in kt.generation_n or relative in kt.generation_n1:
            relatives.append(relative)
    
    return len(relatives)

In [20]:
def data_count(filename: str):
    df = []
    
    codes = []
        
    for code in tqdm(all_glottocodes):

        language = list(kinbank[kinbank['Glottocode'] == code]['Name'])[0] # get the language name
        family = list(kinbank[kinbank['Glottocode'] == code]['Family'])[0] # get the family 

        data = {}

        if code not in codes:
            codes.append(code)

            ks = get_kin_terms(code)
#             print(ks)

            length = count_data(ks)

            results = {}
            results['language'] = language
            results['language_family'] = family
            results['code'] = code
            results['data_length'] = length

            df.append(results)
        
    pd.DataFrame(df).to_csv('../data/new_measure/' + filename + '.csv',index=False)
    
    return pd.DataFrame(df)

In [25]:
data_count('data_count')

100%|██████████████████████████████████████████████████████████████████████████████| 1127/1127 [00:35<00:00, 31.48it/s]


,language,language_family,code,data_length
0,Sirionó,Tupian,siri1273,56
1,Slave,NaN,slav1253,28
2,Carolinian,Austronesian,caro1242,2
3,Kumyk,Turkic,kumy1244,60
4,Nepali,Indo-European,nepa1254,34
...,...,...,...,...
1122,Waropen,Austronesian,waro1242,34
1123,Tetum,Austronesian,tetu1245,32
1124,Piegan,Algic,pieg1239,28
1125,Meyah,NaN,meya1236,28
